# 02 — Feature Engineering
**NSW Property Price Prediction**

This notebook transforms the raw data into model-ready features. Every decision here flows from the EDA findings in `01_eda.ipynb`.

The preprocessing is implemented as a **scikit-learn `Pipeline` + `ColumnTransformer`**.  
This is the industry-standard approach because it:
- Prevents data leakage (fit transformers on train only, then transform both sets)
- Makes the pipeline serialisable and deployable
- Keeps the code clean and reproducible

---
**Sections**
1. Setup & Load Raw Data
2. Initial Cleaning
3. Missing Value Strategy
4. Derived Features
5. Target Encoding (Suburb)
6. Build sklearn Pipeline
7. Train/Test Split & Save

## 1. Setup & Load Raw Data

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sys.path.insert(0, str(Path.cwd().parent))
from src.data_loader import load_raw

RANDOM_STATE = 42  # reproducibility — fix this so train/test split never changes
PROCESSED_DIR = Path.cwd().parent / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Column name configuration — match to your dataset
RAW_FILENAME = "property_sales.csv"
PRICE_COL    = "purchase_price"
SUBURB_COL   = "suburb"
POSTCODE_COL = "postcode"
AREA_COL     = "area"
TYPE_COL     = "nature_of_property"
PURPOSE_COL  = "primary_purpose"
DATE_COL     = "contract_date"

df = load_raw(RAW_FILENAME)
print(f"Loaded {len(df):,} rows")

Loaded 2,202,140 rows


## 2. Initial Cleaning

Decisions made in EDA, applied here:
- Keep residential sales only (`nature_of_property = R`)
- Price floor $50,000: removes data entry errors and non-arm's-length transfers (gifts, family sales) while keeping genuine low-value regional NSW properties
- Price ceiling $30,000,000: removes ~0.1% of extreme outliers; all genuine residential sales fall below this
- **Date filter from 2010 onwards:** Records from 2000–2009 are sparse and, after StandardScaler, produce year z-scores up to 8σ which causes numerical overflow in linear models. 2010+ gives 15 years of data covering multiple market cycles.
- Area unit conversion: some rows record area in hectares (H) — convert to m² for a unified unit
- Area cap at 500,000 m²: removes corrupt values (up to 15 trillion m²) introduced by the hectare conversion

In [3]:
PRICE_MIN  = 50_000
PRICE_MAX  = 30_000_000
DATE_MIN   = "2010-01-01"   # raised from 2000 — records from 2000-2009 produce year
                             # z-scores up to 8σ after StandardScaler, causing linear
                             # model overflow; 2010+ still gives 15 years of market data
AREA_MAX   = 500_000        # 50 hectares — generous upper bound for rural residential

# Filter to residential sales only
df = df[df[TYPE_COL].str.upper().str.startswith("R")].copy()

# Parse dates and drop records outside our window
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df = df[df[DATE_COL] >= DATE_MIN]

# Apply price bounds
df = df[df[PRICE_COL].between(PRICE_MIN, PRICE_MAX)]

# Convert area to numeric, then unify units to m²
# NOTE: area_type "H" means hectares — multiply by 10,000 to get m²
df[AREA_COL] = pd.to_numeric(df[AREA_COL], errors="coerce")
hectare_mask = df["area_type"].str.upper() == "H"
df.loc[hectare_mask, AREA_COL] = df.loc[hectare_mask, AREA_COL] * 10_000

# Cap area at 500,000 m² — removes corrupt source values (up to 15 trillion m²)
df[AREA_COL] = df[AREA_COL].clip(upper=AREA_MAX)

# Extract temporal features after date is clean
df["year"]    = df[DATE_COL].dt.year
df["quarter"] = df[DATE_COL].dt.quarter

# Standardise suburb casing for consistent encoding
df[SUBURB_COL] = df[SUBURB_COL].str.upper().str.strip()

print(f"After cleaning: {len(df):,} rows")
print(f"  Date range: {df[DATE_COL].min().date()} → {df[DATE_COL].max().date()}")
print(f"  Price range: ${df[PRICE_COL].min():,} → ${df[PRICE_COL].max():,}")
print(f"  Area range:  {df[AREA_COL].min():.0f} m² → {df[AREA_COL].max():,.0f} m²")

After cleaning: 1,877,581 rows
  Date range: 2010-01-08 → 2026-05-21
  Price range: $50,000 → $30,000,000
  Area range:  0 m² → 500,000 m²


## 3. Missing Value Strategy

Per-column decisions based on EDA findings:

In [4]:
# Check missingness on our key modelling columns
model_cols = [PRICE_COL, SUBURB_COL, AREA_COL, "year", POSTCODE_COL]
missing = df[model_cols].isnull().sum()
print("Missing values in model columns:")
print(missing[missing > 0].to_string() if (missing > 0).any() else "None")

Missing values in model columns:
suburb          12
area        471334
postcode       535


In [5]:
# Drop rows where target or suburb is missing — no safe imputation for these
df = df.dropna(subset=[PRICE_COL, SUBURB_COL])

# Impute land area with suburb median — better than global median because
# lot sizes vary significantly between suburban and inner-city areas
suburb_median_area = df.groupby(SUBURB_COL)[AREA_COL].transform("median")
df[AREA_COL] = df[AREA_COL].fillna(suburb_median_area)

# Any remaining area nulls (suburbs with all-null area) → global median
df[AREA_COL] = df[AREA_COL].fillna(df[AREA_COL].median())

# Year nulls → median year (rare, only if date parsing failed)
df["year"] = df["year"].fillna(df["year"].median()).astype(int)

print(f"After imputation: {len(df):,} rows")
print(f"Remaining nulls: {df[[PRICE_COL, SUBURB_COL, AREA_COL, 'year']].isnull().sum().sum()}")

After imputation: 1,877,569 rows
Remaining nulls: 0


## 4. Derived Features

New features that may better capture the underlying price drivers.

In [6]:
# Log-transform area: land area is right-skewed and the relationship with
# price is better approximated as log-linear (doubling size ≠ doubling price)
df["log_area"] = np.log1p(df[AREA_COL])

# Log-transform the target — shown in EDA to be strongly right-skewed
# We model log(price), then exponentiate predictions back to AUD for evaluation
df["log_price"] = np.log1p(df[PRICE_COL])

print("Derived features added: log_area, log_price")
df[[AREA_COL, "log_area", PRICE_COL, "log_price"]].describe().round(2)

Derived features added: log_area, log_price


,area,log_area,purchase_price,log_price
count,1877569.00,1877569.00,1877569.00,1877569.00
mean,18896.91,6.23,1150529.56,13.68
std,93863.20,1.62,1306961.88,0.70
min,0.00,0.00,50000.00,10.82
25%,208.60,5.35,590000.00,13.29
50%,520.15,6.26,820000.00,13.62
75%,714.50,6.57,1275000.00,14.06
max,500000.00,13.12,30000000.00,17.22


## 5. Target Encoding (Suburb)

There are too many suburbs for one-hot encoding (would create hundreds of sparse columns).  
**Target encoding** replaces each suburb with its mean log-price computed on the training set.

**Important:** target encoding must be computed *only* on the training set, then applied to the test set. We do this properly here by computing after the train/test split.

In [7]:
# Define features and target
FEATURE_COLS = ["log_area", "year", "quarter", SUBURB_COL]
TARGET_COL   = "log_price"

# We'll encode suburb after splitting, so keep it as a string for now
X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

# 80/20 split — stratification is not straightforward for regression,
# so we use shuffle=True (default) to ensure random distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Train: {len(X_train):,} rows | Test: {len(X_test):,} rows")

Train: 1,502,055 rows | Test: 375,514 rows


In [8]:
# Compute suburb target encoding on TRAIN only, then apply to both sets
# Global mean is the fallback for suburbs that appear only in test set
global_mean = y_train.mean()
suburb_means = X_train.join(y_train).groupby(SUBURB_COL)[TARGET_COL].mean()

X_train["suburb_encoded"] = X_train[SUBURB_COL].map(suburb_means).fillna(global_mean)
X_test["suburb_encoded"]  = X_test[SUBURB_COL].map(suburb_means).fillna(global_mean)

# Clip to ±3σ of the train encoding distribution to prevent extreme suburb mean log-prices
# from causing linear model overflow after StandardScaler (train-only stats prevent leakage)
enc_mean = X_train["suburb_encoded"].mean()
enc_std  = X_train["suburb_encoded"].std()
X_train["suburb_encoded"] = X_train["suburb_encoded"].clip(enc_mean - 3*enc_std, enc_mean + 3*enc_std)
X_test["suburb_encoded"]  = X_test["suburb_encoded"].clip(enc_mean - 3*enc_std, enc_mean + 3*enc_std)

# Drop original suburb string column — model uses the encoded version
X_train = X_train.drop(columns=[SUBURB_COL])
X_test  = X_test.drop(columns=[SUBURB_COL])

print("Final feature columns:", list(X_train.columns))
X_train.head(3)

Final feature columns: ['log_area', 'year', 'quarter', 'suburb_encoded']


,log_area,year,quarter,suburb_encoded
198567,6.143542,2024,3,13.305975
1750082,6.730063,2021,4,14.540064
180864,6.323283,2024,3,13.481963


## 6. Build sklearn Pipeline

The `Pipeline` from `src/preprocessing.py` handles scaling and one-hot encoding for the final set of numerical/categorical features. Since we already target-encoded the suburb and derived log features, the pipeline here is primarily StandardScaler for the numerical inputs.

In [9]:
from src.preprocessing import make_pipeline

# All remaining features are numerical after target encoding
numerical_features = list(X_train.columns)
low_card_cat_features = []  # none remaining after suburb encoding

pipeline = make_pipeline(
    numerical_features=numerical_features,
    low_card_cat_features=low_card_cat_features,
)

# Fit on train, transform both — this is the correct way to prevent leakage
X_train_processed = pipeline.fit_transform(X_train)
X_test_processed  = pipeline.transform(X_test)

print(f"Processed train shape: {X_train_processed.shape}")
print(f"Processed test shape:  {X_test_processed.shape}")

Processed train shape: (1502055, 4)
Processed test shape:  (375514, 4)


## 7. Save Processed Data

Saving processed arrays to `data/processed/` so notebooks 03 and 04 don't need to re-run the full pipeline.

In [10]:
import pickle

# Save as DataFrames to preserve column names (needed for interpretation)
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()

pd.DataFrame(X_train_processed, columns=feature_names).to_parquet(
    PROCESSED_DIR / "X_train.parquet", index=False
)
pd.DataFrame(X_test_processed, columns=feature_names).to_parquet(
    PROCESSED_DIR / "X_test.parquet", index=False
)
y_train.reset_index(drop=True).to_frame().to_parquet(
    PROCESSED_DIR / "y_train.parquet", index=False
)
y_test.reset_index(drop=True).to_frame().to_parquet(
    PROCESSED_DIR / "y_test.parquet", index=False
)

# Save the fitted pipeline for use in notebook 04 (SHAP, predictions)
with open(PROCESSED_DIR / "pipeline.pkl", "wb") as f:
    pickle.dump(pipeline, f)

# Save the suburb encoding map for deployment / future predictions
suburb_means.to_frame(name="suburb_mean_log_price").to_parquet(
    PROCESSED_DIR / "suburb_encoding.parquet"
)

print("Saved to data/processed/:")
print("  X_train.parquet, X_test.parquet")
print("  y_train.parquet, y_test.parquet")
print("  pipeline.pkl")
print("  suburb_encoding.parquet")

Saved to data/processed/:
  X_train.parquet, X_test.parquet
  y_train.parquet, y_test.parquet
  pipeline.pkl
  suburb_encoding.parquet


## Summary

| Step | Decision | Reason |
|------|----------|--------|
| Scope | All NSW residential | Larger dataset; broader market coverage across metro, regional, and rural NSW |
| Price filter | $50k – $30M | Removes data-entry errors and non-arm's-length transfers; retains genuine high-value sales |
| Date filter | 2010 onwards | Pre-2010 records are sparse and produce year z-scores up to 8 sigma after StandardScaler, causing linear model overflow; 2010+ gives 15 years of market data |
| Area units | Hectares x 10,000 to m2 | Unified unit across all rows |
| Area cap | 500,000 m2 | Removes corrupt source values caused by bad hectare conversion (up to 2.7 billion m2) |
| Log target | `log1p(price)` | Right-skewed (skewness 7.32); log-transform gives near-normal residuals |
| Log area | `log1p(area)` | Right-skewed; log-linear relationship with price |
| Suburb encoding | Target encoding (train mean), clipped to +-3 sigma | High cardinality (3,969 suburbs); clipping prevents extreme suburbs destabilising linear models |
| Scaling | `StandardScaler` | Required for Ridge/Lasso regularisation to treat features fairly |
| Train/test split | 80/20, `random_state=42` | Standard split; fixed seed for reproducibility |

Proceed to `03_modelling.ipynb` for model training and comparison.